# How similar are two states? The swap test
**Quantum Machine Learning · Week 1 (Data meets qubits) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qml/w1d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A quantum kernel needs the fidelity $|\langle\psi|\varphi\rangle|^2$ of two states. A quantum computer can't
print amplitudes, but two small circuits turn the fidelity into a measurement probability.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, state_fidelity
from qiskit.primitives import StatevectorSampler

def prep(theta, phi):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return qc

def swap_test(a, b):
    qc = QuantumCircuit(3, 1)
    qc.compose(a, [1], inplace=True)            # psi on qubit 1
    qc.compose(b, [2], inplace=True)            # phi on qubit 2
    qc.h(0)                                     # the ancilla
    qc.cswap(0, 1, 2)
    qc.h(0)
    qc.measure(0, 0)
    return qc

psi, phi = prep(0.6, 0.0), prep(2.0, 1.1)
F = state_fidelity(Statevector(psi), Statevector(phi))
p0 = Statevector(swap_test(psi, phi).remove_final_measurements(inplace=False)).probabilities([0])[0]
print(f"fidelity F = {F:.4f};   P(ancilla = 0) = {p0:.4f};   (1 + F)/2 = {(1 + F) / 2:.4f}")
swap_test(psi, phi).draw("mpl", style="clifford")

## With shots
On hardware we only get counts: $\hat F = 2\,\hat P(0) - 1$.

In [ ]:
sampler = StatevectorSampler(seed=1)
for shots in [100, 1000, 10000]:
    counts = sampler.run([swap_test(psi, phi)], shots=shots).result()[0].data.c.get_counts()
    print(f"{shots:6d} shots: F estimate {2 * counts.get('0', 0) / shots - 1:.4f}   (exact {F:.4f})")

## The inversion test (compute-uncompute)
If we know the circuits that made the states, there's a cheaper way: apply $U(x)$, then $U(x')^\dagger$. The
probability of reading all zeros is exactly the fidelity. No extra qubits, and it's how quantum kernels
are usually measured.

In [ ]:
def zz_map(x, reps=2):
    qc = QuantumCircuit(2)
    for _ in range(reps):
        qc.h([0, 1])
        qc.p(2 * x[0], 0)
        qc.p(2 * x[1], 1)
        qc.cx(0, 1)
        qc.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)
        qc.cx(0, 1)
    return qc

x, x2 = np.array([0.7, 2.1]), np.array([0.9, 1.8])
test = zz_map(x).compose(zz_map(x2).inverse())
test.measure_all()
counts = sampler.run([test], shots=10000).result()[0].data.meas.get_counts()
exact = state_fidelity(Statevector(zz_map(x)), Statevector(zz_map(x2)))
print(f"inversion test: P(00) = {counts.get('00', 0) / 10000:.4f};   exact fidelity {exact:.4f}")

## Your turn: how many shots?
For $|0\rangle$ versus $|+\rangle$ ($F = 0.5$), simulate 2,000 repetitions of each test with $N$ = 100, 1,000 and
10,000 shots (numpy's `rng.binomial` is enough), and compare the spread of the estimates. How many shots
does each test need for a spread of about 0.01?

In [ ]:
rng = np.random.default_rng(0)
F = 0.5
for N in [100, 1000, 10000]:
    swap_estimates = None        # TODO: 2 * rng.binomial(N, (1 + F) / 2, size=2000) / N - 1
    inversion_estimates = None   # TODO: rng.binomial(N, F, size=2000) / N
    # TODO: print N and the two standard deviations

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
rng = np.random.default_rng(0)
F = 0.5
for N in [100, 1000, 10000]:
    swap_estimates = 2 * rng.binomial(N, (1 + F) / 2, size=2000) / N - 1
    inversion_estimates = rng.binomial(N, F, size=2000) / N
    print(f"N = {N:6d}: spread of the swap test {swap_estimates.std():.4f}, inversion test {inversion_estimates.std():.4f}")
print("The swap test multiplies the noise (F = 2P - 1; here by about 1.7), so it needs about 7,500 shots for a spread")
print("of 0.01 here, the inversion test about 2,500. A kernel matrix needs one such estimate for every pair of points.")